# Phase 4 follow-up: template × role interaction + better free-text check (7B, bf16)

Needs the Phase 4 outputs already on Drive (`behavior.csv`). **Same GPU as Phase 4 (A100)**, then `Runtime → Run all`.

1. GPU check, then a **smoke test** of the free-text script on 1 question (throwaway folder).
2. **Stats** again, now with per-template tests, the template × role interaction, and the senior − junior gap per template × pair (CPU, seconds).
3. **Free text** on 50 questions × 13 conditions × **all 3 templates** (≈1,950 replies, a few minutes), with a stricter parser,
   flip/no-flip agreement with the letter metric, and a pronoun-mirroring check. Writes `freetext_sample.csv`:
   30 replies with the question and both answers, **for you to read** (open it from Drive in Sheets).
4. Summary to paste back, then auto-disconnect.

Every step stops `Run all` if it fails.

In [ ]:
CFG = "configs/colab.yaml"   # same config as Phase 4
AUTO_DISCONNECT = True

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/GTRAN7/vnpronouns.git" if token else "https://github.com/GTRAN7/vnpronouns.git"
if not os.path.exists('/content/vnpronouns'):
    subprocess.run(['git', 'clone', '-q', url, '/content/vnpronouns'], check=True)
else:
    subprocess.run(['git', '-C', '/content/vnpronouns', 'pull', '-q'], check=True)
%cd /content/vnpronouns
!git log --oneline -1
!pip install -q -e .

def run(cmd, log, env=None):
    """Run a shell command, stream + log its output, and raise if it fails (so Run all stops).
    Splits only on real newlines and keeps the text after the last carriage return, so tqdm
    progress bars collapse to one line instead of flooding the notebook."""
    print(f'$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1', **(env or {})})
    with open(log, 'a', encoding='utf-8') as f:
        for raw in p.stdout:
            line = raw.decode('utf-8', 'replace').rstrip('\n').split('\r')[-1]
            if line.strip():
                print(line, flush=True)
                f.write(line + '\n')
    if p.wait() != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}  -- see {log}')

## 1. GPU check + smoke test (free text on 1 question, throwaway folder)

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> A100 (or L4)'
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'{name}, {mem:.0f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}')
assert torch.cuda.is_bf16_supported() and mem >= 22, f'{name} cannot hold the 7B in bf16; switch to A100/L4'

import os, shutil, time
OUT = '/content/drive/MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct'
assert os.path.exists(f'{OUT}/behavior.csv'), 'Phase 4 behavior.csv not found on Drive: run colab_7b_behavior.ipynb first'
shutil.rmtree('/content/smoke', ignore_errors=True)
os.makedirs('/content/smoke/Qwen2.5-7B-Instruct')
shutil.copy(f'{OUT}/behavior.csv', '/content/smoke/Qwen2.5-7B-Instruct/behavior.csv')
t0 = time.time()
run(f'python -m vnpronouns.freetext --config {CFG} --n-questions 1', '/content/log_smoke.txt',
    {'VNP_OUTPUT_DIR': '/content/smoke'})
print(f'\nSMOKE TEST PASSED in {time.time() - t0:.0f}s')

## 2. Statistics with the template analysis

In [ ]:
!rm -f /content/log_stats.txt
run(f'python -m vnpronouns.stats --config {CFG}', '/content/log_stats.txt')

## 3. Free text on all 3 templates
Keeps the Phase 4 flat-only file as `freetext_v1_flat.csv`.

In [ ]:
if os.path.exists(f'{OUT}/freetext.csv') and not os.path.exists(f'{OUT}/freetext_v1_flat.csv'):
    shutil.copy(f'{OUT}/freetext.csv', f'{OUT}/freetext_v1_flat.csv')
!rm -f /content/log_freetext.txt
run(f'python -m vnpronouns.freetext --config {CFG} --n-questions 50', '/content/log_freetext.txt')

## 4. Summary to paste back to Claude

In [ ]:
import re, torch
def clean(path):
    lines = open(path, encoding='utf-8').read().splitlines()
    return '\n'.join(l.rstrip() for l in lines
                     if not re.search(r'it/s\]|s/it\]|Warning|warn\(|^\s*$|Loading|Fetching|Downloading|%\||^\$ ', l))
print(f'GPU: {torch.cuda.get_device_name(0)}  config: {CFG}')
for p in ['/content/log_stats.txt', '/content/log_freetext.txt']:
    print(f'===== {p.split("/")[-1]} =====')
    print(clean(p))
!ls -la /content/drive/MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct/

In [ ]:
if AUTO_DISCONNECT:
    from google.colab import runtime
    print('Done. Disconnecting the runtime to save compute units (outputs above stay visible).')
    runtime.unassign()